# bias-correction-divide composite — cx24: Adam ratio: m_hat / (sqrt(v_hat) + eps)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `bias-correction-divide`, `sqrt-eps-stabilize`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "bias-correction-divide"
DD_ATOM_IDS = ["bias-correction-divide", "sqrt-eps-stabilize"]
DD_SUBTOPICS = ["Optimizer: Adam bias-correction divide", "Numerical: sqrt-eps stabilization"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Adam's full parameter update is `p <- p - lr * step` where the step is built from TWO bias-corrected moments and the sqrt-eps denominator:
```
step = m_hat / (sqrt(v_hat) + eps)
     = ( m / (1 - beta1**t) )                         <- bias-correction-divide (atom A)
       / ( sqrt( v / (1 - beta2**t) ) + eps )         <- sqrt-eps-stabilize (atom B)
```
**Both atoms are bias-correction-divide and sqrt-eps-stabilize.** Atom A is invoked TWICE in the equation (once for `m_hat`, once inside the sqrt for `v_hat`), and atom B wraps the result. The composition is the ratio that gives Adam its adaptive, scale-invariant per-coordinate step.

**Why this exact form is scale-invariant.** Multiply `g` by 10. Then `m` scales by 10, and `v` scales by 100, so `sqrt(v_hat)` scales by 10 — the ratio is invariant. That's the whole reason Adam is so robust to gradient magnitude across layers.

**Eps placement matters AT SCALE.** With eps-outside (Adam convention), the ratio at small `v_hat` becomes `m_hat / eps` — a finite-but-large number. With eps-inside (BatchNorm convention), the ratio is `m_hat / sqrt(eps)` — much smaller. Adam wants the MORE AGGRESSIVE step in tiny-`v` regimes, so eps-outside is correct.

**Anatomy.**
```python
def adam_ratio(m, v, beta1, beta2, t, eps):
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    return m_hat / (v_hat.sqrt() + eps)
```

### Composite Exercise — Adam ratio: m_hat / (sqrt(v_hat) + eps)

**Atoms exercised together**: `bias-correction-divide`, `sqrt-eps-stabilize`

Implement `cx24_adam_ratio(m, v, beta1, beta2, t_step, eps)`.

Inputs:
- `m`: first-moment buffer (Tensor).
- `v`: second-moment buffer (Tensor, non-negative elementwise — caller's responsibility).
- `beta1`, `beta2`: floats.
- `t_step`: int >= 1.
- `eps`: float (typically 1e-8).

Steps:
1. `m_hat = m / (1 - beta1 ** t_step)` (atom: bias-correction-divide).
2. `v_hat = v / (1 - beta2 ** t_step)` (atom: bias-correction-divide — second use).
3. `denom = sqrt(v_hat) + eps` (atom: sqrt-eps-stabilize — eps OUTSIDE).
4. Return `m_hat / denom` (a fresh tensor, same shape as `m`).

Do not mutate `m` or `v`. Cross-check the full-step equation vs `torch.optim.Adam`'s reference, since this is the EXACT ratio Adam multiplies by `-lr`.

Tests verify:
- `m=0, v=0 -> ratio=0` (zero numerator, finite denominator from eps).
- Sign of the ratio matches sign of `m` elementwise (denominator is always positive).
- Scale-invariance: multiplying `(m, v)` by `(c, c**2)` leaves the ratio unchanged.
- Cross-check: replicates Adam's update direction (`-step`) from `torch.optim.Adam`.
- Eps placement: with `m=1, v=0`, ratio == `1 / eps` (eps outside), NOT `1 / sqrt(eps)`.

In [ ]:
def cx24_adam_ratio(m, v, beta1, beta2, t_step, eps):
    # Atom A (bias-correction-divide): applied to m...
    m_hat = m / (1.0 - beta1 ** t_step)
    # ...and to v (same atom, second invocation, using beta2).
    v_hat = v / (1.0 - beta2 ** t_step)
    # Atom B (sqrt-eps-stabilize): Adam convention — eps OUTSIDE the sqrt.
    denom = v_hat.sqrt() + eps
    return m_hat / denom


<details><summary>Show solution — cx24</summary>

```python
def cx24_adam_ratio(m, v, beta1, beta2, t_step, eps):
    # Atom A (bias-correction-divide): applied to m...
    m_hat = m / (1.0 - beta1 ** t_step)
    # ...and to v (same atom, second invocation, using beta2).
    v_hat = v / (1.0 - beta2 ** t_step)
    # Atom B (sqrt-eps-stabilize): Adam convention — eps OUTSIDE the sqrt.
    denom = v_hat.sqrt() + eps
    return m_hat / denom
```

**This is the load-bearing line of Adam.** Everything else (m/v EMAs, beta defaults, lr schedule, weight decay) is decoration around this ratio. Getting the ratio's structure wrong is what makes a from-scratch Adam train slower than `torch.optim.Adam`.

**Two atoms, three operations.** The bias-correction-divide atom is invoked TWICE in the same expression — once on `m`, once on `v` — because both moments need de-biasing at the same step `t`. The sqrt-eps-stabilize atom wraps the v-side result. This is the most multiply-invoked atom composition in the entire Adam step.

**Scale invariance is the diagnostic.** If your ratio depends on the absolute magnitude of `g` (not just its direction relative to history), you've broken the bias correction or moved eps inside the sqrt. Case C is the cleanest invariant: scaling `(m, v)` by `(c, c**2)` MUST leave the ratio invariant (for `eps << sqrt(v_hat)`).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx24'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx24',
        'subtopics': ["Optimizer: Adam bias-correction divide", "Numerical: sqrt-eps stabilization"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()